# Paired bank Δ, not paper Elo — why 20–0 is not 3200

destbreso already wrote the better version of this argument (*Six checks*, *Compare two agents*). I am adding the number that burned us, and asking for a dual-safe way to certify a 2500+ settle.

## Two scores exist

| Score | What it is | What 3200 means |
|-------|------------|-----------------|
| Live last-2 Glicko | public gym | a file that *settles* ≥3200 in latest-2 |
| Prize Bradley-Terry | +14 days after 2026-09-30 | a policy that still plays after the cassette field moves |

Host topic 731587 is the prize. Topic 734000 is the gym (byte-identical bots 1700 vs 3000, two hours apart).

## The invert that lies (term717)

Wilson / Elo invert

\[
R = R_{\mathrm{live}} - 400\log_{10}\frac{1-p}{p}
\]

is only a paper number. If you invert a 20–0 against a **dump** live rating (~1300) you can print 3700. The same file against live 2985–3143 **win** tapes printed R_lo **3085–3202** (spread 117). Including the dump anchor drops R_lo_min to **1603**. Spread 1482 > 400. We keep the **lower** number and refuse the submit.

destbreso's ladder test of the same idea: offline pairing called the **order** of two routes and missed the **level** by 24–31 win-rate points. Quote a paired difference. Do not quote a pool win rate as a forecast.

## What 1.32.7 did **not** move

destbreso *What 1.32.7 changed*: milk, wool, wheat, straw, melon, fertilizer curves are untouched. Carrot got an extra `below_target` 0.20→1.00. Median game: almost nothing. Deepest carrot scarcity: ~9.6×.

So cell A (milk) and cell B (wool) are **not** hinge-PI bugs. They were already in 1.32.6.

## Six checks I now treat as a gate

From destbreso, in order:

1. Same engine — replay a known bank, do not trust the version string.
2. Flag off ⇒ byte-identical parent.
3. Survives a season (invalid actions are silent no-ops).
4. How many **distinct** opponents (40 names can be 2 farm plans).
5. Identical games, then difference (sign test / median), not two separate WR.
6. Do not keep the best of N knobs and quote that score.

Our shop-latch line (W43–W49) was check 6 waiting to happen. It is frozen.

## Live board this morning (2026-08-18)

| rank | team | score |
|-----:|------|------:|
| 1 | カワシギ | 3188 |
| 2 | Thomas Tschinkel | 3154 |
| 3 | tetsuya | 3044 |
| 4 | Matteo123383iend | 3023 |
| 30 | (cut) | ~2829 |
| us | Kerem çelik | **1426** (dump dual, untouched) |

New #1-package games overnight are still almost all vs Thomas. Newest Thomas **win** tapes exist (ep 94135297 bank 95129; ep 94079899 bank 91725). Newest #1 **win** 103k is ep **94121252**.

tetsuya is still missing from the public pairings I can see. If you have one, I will distill only a **win**.

## What I will not do

- Submit the lab file "to measure 2500" (dual goes cold ~600; team max stays 1426 only if we keep the dump).
- Invert 17–3 vs peikopon as a 3100 settle.
- Open another shop-boolean.

## What I will do if the community has a tape

- L2 the frozen file against the two newest #1 **wins**.
- One destbreso-legal market isolate (already queued: 94121252 market from 193).
- Recert L3 only against live ≥2500 **win** tapes, and only if dump-anchor spread < 400.

Please argue with the invert, the two fail cells, or the 193 donor. I will update the notebook.



In [ ]:
import json, math, os, subprocess, sys
from pathlib import Path

def ensure_env(min_ver=(1, 32, 7)):
    try:
        from importlib.metadata import version
        inst = version("kaggle-environments")
    except Exception:
        inst = "0"
    parts = []
    for p in inst.split("."):
        try:
            parts.append(int(p))
        except ValueError:
            break
    while len(parts) < 3:
        parts.append(0)
    if tuple(parts[:3]) < min_ver:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle-environments>=1.32.7"])
    import kaggle_environments
    print("kaggle-environments", kaggle_environments.__version__)
    return kaggle_environments

ke = ensure_env()
from kaggle_environments import make
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
WORK.mkdir(exist_ok=True)
print("work", WORK)


In [ ]:
def wilson_lo(w, n, z=1.96):
    if n <= 0:
        return 0.0
    p = w / n
    den = 1 + z * z / n
    centre = p + z * z / (2 * n)
    margin = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (centre - margin) / den

def invert(p, Ra):
    p = min(max(p, 1e-6), 1 - 1e-6)
    return Ra - 400 * math.log10((1 - p) / p)

rows = [
    ("Thomas@3143 16-4", 16, 20, 3143),
    ("Thomas@3143 14-6", 14, 20, 3143),
    ("peikopon@2985 17-3", 17, 20, 2985),
    ("kaito dump@1316 20-0", 20, 20, 1316),
]
print(f"{'anchor':22} {'WR':6} {'R_hat':8} {'R_lo':8}")
out = []
for name, w, n, ra in rows:
    p = w / n
    plo = wilson_lo(w, n)
    rec = {"name": name, "W": w, "n": n, "R_hat": round(invert(p, ra), 1), "R_lo": round(invert(plo, ra), 1)}
    out.append(rec)
    print(f"{name:22} {p:5.2f} {rec['R_hat']:8.1f} {rec['R_lo']:8.1f}")

rlo = [r["R_lo"] for r in out]
card = {
    "R_lo_win_tapes": rlo[:3],
    "R_lo_min_with_dump": min(rlo),
    "spread_with_dump": round(max(rlo) - min(rlo), 1),
    "term717": (max(rlo) - min(rlo)) > 400,
    "submit": False,
}
(WORK / "evidence_paired_not_paper.json").write_text(json.dumps({"rows": out, "card": card}, indent=2))
print(json.dumps(card, indent=2))

